In [1]:
# 1 — KAGGLE ENVIRONMENT + GLOBAL PATH REGISTRY (FINAL)

import os
import gc
import numpy as np
import pandas as pd

# --- FIND COMPETITION ROOT ---
BASE_CANDIDATES = [
    "/kaggle/input/competitions/stanford-rna-3d-folding-2",
    "/kaggle/input/stanford-rna-3d-folding-2",
]

COMP_ROOT = None
for p in BASE_CANDIDATES:
    if os.path.exists(p):
        COMP_ROOT = p
        break

if COMP_ROOT is None:
    raise FileNotFoundError(
        "❌ Competition dataset not found.\n"
        "👉 Click 'Add Input' and attach: Stanford RNA 3D Folding 2"
    )

# --- INPUT FILES ---
TRAIN_LABELS_PATH = os.path.join(COMP_ROOT, "train_labels.csv")
VALIDATION_LABELS_PATH = os.path.join(COMP_ROOT, "validation_labels.csv")
TRAIN_SEQS_PATH = os.path.join(COMP_ROOT, "train_sequences.csv")
VALIDATION_SEQS_PATH = os.path.join(COMP_ROOT, "validation_sequences.csv")
TEST_SEQS_PATH = os.path.join(COMP_ROOT, "test_sequences.csv")
SAMPLE_SUB_PATH = os.path.join(COMP_ROOT, "sample_submission.csv")

REQUIRED_FILES = [
    TRAIN_LABELS_PATH,
    VALIDATION_LABELS_PATH,
    TRAIN_SEQS_PATH,
    VALIDATION_SEQS_PATH,
    TEST_SEQS_PATH,
    SAMPLE_SUB_PATH,
]

missing_files = [p for p in REQUIRED_FILES if not os.path.exists(p)]
if missing_files:
    raise FileNotFoundError("❌ Missing required files:\n" + "\n".join(missing_files))

# --- WORKING OUTPUT PATHS ---
WORK_DIR = "/kaggle/working"
os.makedirs(WORK_DIR, exist_ok=True)

FEATURE_OUT = os.path.join(WORK_DIR, "FEATURE_TABLE__GEOMETRY_LABELS_V1.csv")
BENCH_OUT   = os.path.join(WORK_DIR, "HELIX_GEOMETRY_RECON_BENCHMARK_V1.csv")

# --- PRINT STATE ---
print("✅ Competition root:", COMP_ROOT)
print("✅ Working dir:", WORK_DIR)
print("✅ Feature output:", FEATURE_OUT)
print("✅ Benchmark output:", BENCH_OUT)

print("\n📁 Files in competition root:")
for f in os.listdir(COMP_ROOT):
    print(" -", f)

✅ Competition root: /kaggle/input/competitions/stanford-rna-3d-folding-2
✅ Working dir: /kaggle/working
✅ Feature output: /kaggle/working/FEATURE_TABLE__GEOMETRY_LABELS_V1.csv
✅ Benchmark output: /kaggle/working/HELIX_GEOMETRY_RECON_BENCHMARK_V1.csv

📁 Files in competition root:
 - MSA
 - sample_submission.csv
 - validation_sequences.csv
 - test_sequences.csv
 - validation_labels.csv
 - extra
 - train_labels.csv
 - train_sequences.csv
 - PDB_RNA


In [2]:
# 2 — LOAD COMPETITION LABELS (MEMORY-SAFE, OPTIMIZED)

import pandas as pd
import numpy as np
import gc

USE_COLS = ["ID", "resname", "resid", "x_1", "y_1", "z_1", "chain", "copy"]

DTYPES = {
    "ID": "string",
    "resname": "category",
    "resid": "int32",
    "x_1": "float32",
    "y_1": "float32",
    "z_1": "float32",
    "chain": "category",
    "copy": "category",
}

print("📥 Loading train labels...")
df_labels = pd.read_csv(
    TRAIN_LABELS_PATH,
    usecols=USE_COLS,
    dtype=DTYPES,
    low_memory=False
)

print("📥 Appending validation labels...")
df_val = pd.read_csv(
    VALIDATION_LABELS_PATH,
    usecols=USE_COLS,
    dtype=DTYPES,
    low_memory=False
)

# Append WITHOUT creating large intermediate copies
df_labels = pd.concat([df_labels, df_val], ignore_index=True)
del df_val
gc.collect()

# --- DERIVE FIELDS ---
df_labels["target_id"] = df_labels["ID"].str.split("_").str[0]

df_labels["residue_index"] = df_labels["resid"].astype(np.int32)

df_labels["x"] = df_labels["x_1"]
df_labels["y"] = df_labels["y_1"]
df_labels["z"] = df_labels["z_1"]

# Drop original heavy columns EARLY
df_labels = df_labels.drop(columns=["ID", "resid", "x_1", "y_1", "z_1"])

gc.collect()

# --- SORT (critical for geometry) ---
df_labels = df_labels.sort_values(
    ["target_id", "chain", "copy", "residue_index"]
).reset_index(drop=True)

gc.collect()

# --- FINAL REPORT ---
print("✅ Label rows loaded:", len(df_labels))
print("✅ Targets:", df_labels["target_id"].nunique())
print("✅ Chains/copies:", df_labels.groupby(["target_id", "chain", "copy"]).ngroups)

print("\n📊 Memory usage (MB):", round(df_labels.memory_usage(deep=True).sum() / 1e6, 2))

df_labels.head(10)

📥 Loading train labels...
📥 Appending validation labels...
✅ Label rows loaded: 7804733
✅ Targets: 5744
✅ Chains/copies: 17952

📊 Memory usage (MB): 1330.14


,resname,chain,copy,target_id,residue_index,x,y,z
0,C,A,1,157D,1,4.843000,-5.640,13.265
1,G,A,1,157D,2,3.385000,-7.613,8.267
2,C,A,1,157D,3,2.158000,-6.751,2.949
3,G,A,1,157D,4,2.669000,-4.843,-1.773
4,A,A,1,157D,5,3.509000,0.239,-4.045
5,A,A,1,157D,6,6.073000,4.823,-5.035
6,U,A,1,157D,7,10.129000,7.706,-4.251
7,U,A,1,157D,8,15.514000,8.745,-2.055
8,A,A,1,157D,9,20.429001,7.281,-1.699
9,G,A,1,157D,10,23.509001,2.728,-1.918


In [3]:
# 3 — ENFORCE SEQUENTIAL GEOMETRY + PHYSICAL STEP FILTER

group_cols = ["target_id", "chain", "copy"]

df_geo = df_labels.copy()

df_geo["resid_diff"] = df_geo.groupby(group_cols)["residue_index"].diff()
df_geo["is_seq"] = df_geo["resid_diff"].eq(1)

df_geo = df_geo[df_geo["resid_diff"].isna() | df_geo["is_seq"]].copy()

df_geo["dx"] = df_geo.groupby(group_cols)["x"].diff()
df_geo["dy"] = df_geo.groupby(group_cols)["y"].diff()
df_geo["dz"] = df_geo.groupby(group_cols)["z"].diff()
df_geo["step"] = np.sqrt(df_geo["dx"]**2 + df_geo["dy"]**2 + df_geo["dz"]**2)

df_geo = df_geo[
    df_geo["step"].isna() | ((df_geo["step"] >= 4.5) & (df_geo["step"] <= 8.0))
].copy()

steps = df_geo["step"].dropna().values
print("✅ Sequential geometry rows:", len(df_geo))
print("✅ Step mean:", float(steps.mean()))
print("✅ Step median:", float(np.median(steps)))
print("✅ Step p90:", float(np.percentile(steps, 90)))
print("✅ Step min:", float(steps.min()))
print("✅ Step max:", float(steps.max()))

display(df_geo.head(10))


✅ Sequential geometry rows: 6996848
✅ Step mean: 5.791539669036865
✅ Step median: 5.574956893920898
✅ Step p90: 7.013174057006836
✅ Step min: 4.500007629394531
✅ Step max: 7.999997615814209


,resname,chain,copy,target_id,residue_index,x,y,z,resid_diff,is_seq,dx,dy,dz,step
0,C,A,1,157D,1,4.843000,-5.640,13.265,NaN,False,NaN,NaN,NaN,NaN
1,G,A,1,157D,2,3.385000,-7.613,8.267,1.0,True,-1.458000,-1.973,-4.998,5.567629
2,C,A,1,157D,3,2.158000,-6.751,2.949,1.0,True,-1.227000,0.862,-5.318,5.525369
3,G,A,1,157D,4,2.669000,-4.843,-1.773,1.0,True,0.511000,1.908,-4.722,5.118483
4,A,A,1,157D,5,3.509000,0.239,-4.045,1.0,True,0.840000,5.082,-2.272,5.629770
5,A,A,1,157D,6,6.073000,4.823,-5.035,1.0,True,2.564000,4.584,-0.990,5.344834
6,U,A,1,157D,7,10.129000,7.706,-4.251,1.0,True,4.056000,2.883,0.784,5.037606
7,U,A,1,157D,8,15.514000,8.745,-2.055,1.0,True,5.385000,1.039,2.196,5.907636
8,A,A,1,157D,9,20.429001,7.281,-1.699,1.0,True,4.915001,-1.464,0.356,5.140746
9,G,A,1,157D,10,23.509001,2.728,-1.918,1.0,True,3.080000,-4.553,-0.219,5.501288


In [4]:
# 4 — BUILD GEOMETRY FEATURES (NORMALIZED DIRECTION + CURVATURE + TURN ANGLE)

df_feat = df_geo.copy()

df_feat["step_safe"] = df_feat["step"].replace(0, np.nan)
df_feat["dx_norm"] = df_feat["dx"] / df_feat["step_safe"]
df_feat["dy_norm"] = df_feat["dy"] / df_feat["step_safe"]
df_feat["dz_norm"] = df_feat["dz"] / df_feat["step_safe"]

df_feat["dx_prev"] = df_feat.groupby(group_cols)["dx"].shift(1)
df_feat["dy_prev"] = df_feat.groupby(group_cols)["dy"].shift(1)
df_feat["dz_prev"] = df_feat.groupby(group_cols)["dz"].shift(1)
df_feat["step_prev"] = df_feat.groupby(group_cols)["step"].shift(1)

df_feat["dx_prev_norm"] = df_feat["dx_prev"] / df_feat["step_prev"]
df_feat["dy_prev_norm"] = df_feat["dy_prev"] / df_feat["step_prev"]
df_feat["dz_prev_norm"] = df_feat["dz_prev"] / df_feat["step_prev"]

df_feat["curvature"] = (
    df_feat["dx_norm"] * df_feat["dx_prev_norm"] +
    df_feat["dy_norm"] * df_feat["dy_prev_norm"] +
    df_feat["dz_norm"] * df_feat["dz_prev_norm"]
)
df_feat["curvature"] = np.clip(df_feat["curvature"], -1.0, 1.0)
df_feat["turn_angle"] = np.arccos(df_feat["curvature"])

feature_cols = ["dx_norm", "dy_norm", "dz_norm", "curvature", "turn_angle", "step"]

df_feat = df_feat.dropna(subset=feature_cols).reset_index(drop=True)
df_feat.to_csv(FEATURE_OUT, index=False)

print("✅ Feature table rows:", len(df_feat))
print("✅ Feature columns:", feature_cols)
print("✅ Saved:", FEATURE_OUT)

display(df_feat[feature_cols].describe())


✅ Feature table rows: 6441646
✅ Feature columns: ['dx_norm', 'dy_norm', 'dz_norm', 'curvature', 'turn_angle', 'step']
✅ Saved: /kaggle/working/FEATURE_TABLE__GEOMETRY_LABELS_V1.csv


,dx_norm,dy_norm,dz_norm,curvature,turn_angle,step
count,6.441646e+06,6.441646e+06,6.441646e+06,6.441646e+06,6.441646e+06,6.441646e+06
mean,-1.177979e-03,-4.055961e-04,5.236998e-05,6.556832e-01,7.731258e-01,5.791190e+00
std,5.763636e-01,5.767268e-01,5.749628e-01,4.014457e-01,4.991009e-01,7.265077e-01
min,-9.999998e-01,-1.000000e+00,-9.999996e-01,-9.999973e-01,0.000000e+00,4.500008e+00
25%,-5.014891e-01,-5.015000e-01,-4.989068e-01,6.167521e-01,4.604168e-01,5.306649e+00
50%,-3.073370e-03,-1.161220e-03,-1.881477e-04,8.187188e-01,6.116201e-01,5.574335e+00
75%,4.980182e-01,4.998140e-01,4.992287e-01,8.958674e-01,9.061865e-01,6.020129e+00
max,9.999999e-01,1.000000e+00,9.999996e-01,1.000000e+00,3.139277e+00,7.999998e+00


In [5]:
# 5 — LOAD GEOMETRY FEATURE TABLE (DATASET — FINAL, MEMORY SAFE)

import pandas as pd
import gc
import os

# --- DATASET PATH (CONFIRMED STRUCTURE) ---
FEATURE_PATH = "/kaggle/input/datasets/pharaohtut/helix-rna-geometry-v1/FEATURE_TABLE__GEOMETRY_FULL.parquet"

if not os.path.exists(FEATURE_PATH):
    raise FileNotFoundError(f"❌ Feature table not found at:\n{FEATURE_PATH}")

print("📥 Loading geometry feature table...")
print("Path:", FEATURE_PATH)

# --- LOAD PARQUET (COLUMN-OPTIMIZED) ---
USE_COLS = [
    "target_id",
    "chain",
    "copy",
    "residue_index",
    "x", "y", "z",
    "dx", "dy", "dz",
    "dx_norm", "dy_norm", "dz_norm",
    "curvature",
    "turn_angle",
    "step"
]

df_feat = pd.read_parquet(FEATURE_PATH, columns=USE_COLS)

# --- TYPE OPTIMIZATION (CRITICAL FOR MEMORY) ---
float_cols = [
    "x","y","z",
    "dx","dy","dz",
    "dx_norm","dy_norm","dz_norm",
    "curvature","turn_angle","step"
]

for col in float_cols:
    df_feat[col] = df_feat[col].astype("float32")

df_feat["residue_index"] = df_feat["residue_index"].astype("int32")

# --- SORT FOR SEQUENTIAL OPERATIONS ---
df_feat = df_feat.sort_values(
    ["target_id", "chain", "copy", "residue_index"]
).reset_index(drop=True)

# --- BASIC VALIDATION ---
print("\n✅ Feature table loaded")
print("Rows:", len(df_feat))
print("Targets:", df_feat["target_id"].nunique())
print("Chains:", df_feat.groupby(["target_id","chain","copy"]).ngroups)

print("\n📊 Sample:")
display(df_feat.head())

gc.collect()

📥 Loading geometry feature table...
Path: /kaggle/input/datasets/pharaohtut/helix-rna-geometry-v1/FEATURE_TABLE__GEOMETRY_FULL.parquet

✅ Feature table loaded
Rows: 7768829
Targets: 5744
Chains: 17746

📊 Sample:


,target_id,chain,copy,residue_index,x,y,z,dx,dy,dz,dx_norm,dy_norm,dz_norm,curvature,turn_angle,step
0,157D,A,1,2,3.385,-7.613,8.267,-1.458,-1.973,-4.998,-0.261871,-0.354370,-0.897689,0.521913,0.521913,5.567629
1,157D,A,1,3,2.158,-6.751,2.949,-1.227,0.862,-5.318,-0.222067,0.156008,-0.962470,0.392644,0.392644,5.525369
2,157D,A,1,4,2.669,-4.843,-1.773,0.511,1.908,-4.722,0.099834,0.372767,-0.922539,0.761646,0.761646,5.118483
3,157D,A,1,5,3.509,0.239,-4.045,0.840,5.082,-2.272,0.149207,0.902701,-0.403569,0.401361,0.401361,5.629770
4,157D,A,1,6,6.073,4.823,-5.035,2.564,4.584,-0.990,0.479716,0.857651,-0.185226,0.558137,0.558137,5.344834


0

In [6]:
# 11 — HELIX_V08_DISTANCE_SHAPED_COMPACT_EXPANSIVE

import hashlib
import numpy as np
import pandas as pd

sample = pd.read_csv(SAMPLE_SUB_PATH)
test_df = pd.read_csv(TEST_SEQS_PATH)

ENSEMBLE_SIZE = 5
TARGET_STEP = np.float32(5.82)
EPS = np.float32(1e-8)

def normalize(v):
    v = np.asarray(v, dtype=np.float32)
    n = np.linalg.norm(v)
    if n < EPS:
        return np.array([1.0, 0.0, 0.0], dtype=np.float32)
    return v / n

def seed(text):
    return int(hashlib.md5(text.encode()).hexdigest()[:8], 16)

target_structures = {}

for _, row in test_df.iterrows():
    target_id = row["target_id"]
    seq = row["sequence"]
    L = len(seq)

    ensemble_coords = []

    for m in range(ENSEMBLE_SIZE):
        rng = np.random.default_rng(seed(f"{target_id}_{m}"))

        coords = np.zeros((L,3), dtype=np.float32)

        direction = normalize(np.array([1.0, 0.25*m, 0.15*(m-2)], dtype=np.float32))
        phase = rng.uniform(0, 2*np.pi)

        for i in range(1, L):

            # --- CURVATURE BACKBONE ---
            ortho = normalize(np.cross(direction, np.array([0,0,1], dtype=np.float32)))
            twist = normalize(np.cross(direction, ortho))

            curve = (
                np.sin(phase) * ortho +
                0.35 * np.cos(phase * 0.7) * twist
            )

            direction = normalize(0.90 * direction + 0.55 * curve)

            # --- STEP ---
            coords[i] = coords[i-1] + direction * TARGET_STEP

            # 🔥 DISTANCE SHAPING
            if i > 8:

                # --- SHORT RANGE ATTRACTOR ---
                k1 = 2 + (i % 3)
                alpha1 = 0.10 + 0.02 * m

                # --- MID RANGE ATTRACTOR ---
                k2 = 5 + (i % 4)
                alpha2 = 0.06 + 0.015 * m

                # --- REPULSION (prevents collapse) ---
                k3 = 9 + (i % 5)
                beta = 0.07 + 0.02 * m

                # --- APPLY FORCES ---
                coords[i] += alpha1 * (coords[i-k1] - coords[i])
                coords[i] += alpha2 * (coords[i-k2] - coords[i])
                coords[i] += beta  * (coords[i] - coords[i-k3])

                # --- PHASE MODULATION ---
                coords[i] += 0.03 * np.sin(phase) * ortho

                # --- STEP RENORMALIZATION ---
                step_vec = coords[i] - coords[i-1]
                step_vec = normalize(step_vec) * TARGET_STEP
                coords[i] = coords[i-1] + step_vec

            phase += 0.45 + 0.12 * m

        ensemble_coords.append(coords)

    target_structures[target_id] = ensemble_coords

# --- BUILD FROM SAMPLE ---
submission = sample.copy()

coord_cols = [c for c in submission.columns if c.startswith(("x_", "y_", "z_"))]
submission[coord_cols] = submission[coord_cols].astype(np.float32)

for idx, row in submission.iterrows():
    target_id, resid = row["ID"].rsplit("_", 1)
    resid = int(resid) - 1

    for m in range(ENSEMBLE_SIZE):
        xyz = target_structures[target_id][m][resid]
        submission.at[idx, f"x_{m+1}"] = xyz[0]
        submission.at[idx, f"y_{m+1}"] = xyz[1]
        submission.at[idx, f"z_{m+1}"] = xyz[2]

print("🔥 HELIX_V08 submission ready")
display(submission.head())

🔥 HELIX_V08 submission ready


,ID,resname,resid,x_1,y_1,z_1,x_2,y_2,z_2,x_3,y_3,z_3,x_4,y_4,z_4,x_5,y_5,z_5
0,8ZNQ_1,A,1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,8ZNQ_2,C,2,5.683475,-0.626329,-1.085461,4.125250,4.094403,0.300969,5.685894,0.675713,-1.042312,2.052566,5.196061,1.631051,4.740595,2.970800,1.604216
2,8ZNQ_3,C,3,11.492998,-0.332340,-1.273654,6.057983,9.460283,1.460398,10.782571,-1.685783,-2.565347,1.324294,10.482731,3.953365,8.010289,7.110125,4.063380
3,8ZNQ_4,G,4,16.720688,2.077973,-0.416882,6.915150,14.948735,3.196648,14.013996,-6.286756,-4.069120,-0.005932,15.418295,6.736009,8.147571,12.089678,7.072968
4,8ZNQ_5,U,5,19.883259,6.627439,1.364383,8.653522,20.179882,5.063616,15.987498,-11.652781,-5.156991,0.617642,20.640835,9.227720,6.106944,16.287659,10.549346


In [7]:
# 12 — VALIDATE AND SAVE FINAL SUBMISSION (STRICT — SCHEMA LOCKED)

import numpy as np
import pandas as pd

OUT_PATH = "/kaggle/working/submission.csv"

print("🔍 Running strict submission validation...")

sample = pd.read_csv(SAMPLE_SUB_PATH)

# --- HARD SCHEMA LOCK ---
expected_cols = list(sample.columns)

missing_cols = [c for c in expected_cols if c not in submission.columns]
if missing_cols:
    raise ValueError(f"❌ Submission is missing required columns: {missing_cols}")

extra_cols = [c for c in submission.columns if c not in expected_cols]
if extra_cols:
    print(f"⚠️ Dropping extra columns not allowed by sample_submission: {extra_cols}")

# Keep only exact sample columns and exact order
submission = submission.loc[:, expected_cols].copy()

# --- BASIC STRUCTURE CHECKS ---
if list(submission.columns) != expected_cols:
    raise ValueError(
        "❌ Column mismatch after schema lock\n\n"
        f"Expected:\n{expected_cols}\n\n"
        f"Got:\n{list(submission.columns)}"
    )

if len(submission) != len(sample):
    raise ValueError(
        f"❌ Row count mismatch: expected {len(sample)}, got {len(submission)}"
    )

# --- ID VALIDATION ---
sub_ids = submission["ID"].astype(str).str.strip().reset_index(drop=True)
sample_ids = sample["ID"].astype(str).str.strip().reset_index(drop=True)

if not sub_ids.equals(sample_ids):
    mismatches = (sub_ids != sample_ids)
    mismatch_indices = np.where(mismatches)[0][:10]

    debug_rows = pd.DataFrame({
        "submission_ID": sub_ids.iloc[mismatch_indices].tolist(),
        "sample_ID": sample_ids.iloc[mismatch_indices].tolist(),
    })

    raise ValueError(
        "❌ Submission IDs do NOT match sample_submission.csv\n\n"
        f"First mismatches:\n{debug_rows}"
    )

print("✅ ID alignment PASSED")

# --- COORDINATE VALIDATION ---
coord_cols = [c for c in submission.columns if c.startswith(("x_", "y_", "z_"))]

if len(coord_cols) == 0:
    raise ValueError("❌ No coordinate columns found (x_*, y_*, z_*)")

# Force numeric before validation
for c in coord_cols:
    submission[c] = pd.to_numeric(submission[c], errors="raise")

if submission[coord_cols].isnull().any().any():
    bad = submission[coord_cols].isnull().sum()
    raise ValueError(f"❌ Null values detected:\n{bad[bad > 0]}")

vals = submission[coord_cols].to_numpy(dtype=np.float64)
if not np.isfinite(vals).all():
    raise ValueError("❌ Non-finite values found (NaN or Inf)")

print("✅ Coordinate validity PASSED")

# --- TYPE ENFORCEMENT ---
submission["resname"] = submission["resname"].astype(str)
submission["resid"] = pd.to_numeric(submission["resid"], errors="raise").astype(np.int32)
for c in coord_cols:
    submission[c] = submission[c].astype(np.float32)

print("✅ Dtype enforcement PASSED")

# --- FINAL SAVE ---
submission.to_csv(OUT_PATH, index=False)

print("\n🚀 SUBMISSION READY")
print("📁 Saved to:", OUT_PATH)
print("📐 Shape:", submission.shape)
print("📊 Coordinate dtype sample:")
print(submission[coord_cols].dtypes.head())

🔍 Running strict submission validation...
✅ ID alignment PASSED
✅ Coordinate validity PASSED
✅ Dtype enforcement PASSED

🚀 SUBMISSION READY
📁 Saved to: /kaggle/working/submission.csv
📐 Shape: (9762, 18)
📊 Coordinate dtype sample:
x_1    float32
y_1    float32
z_1    float32
x_2    float32
y_2    float32
dtype: object
